# NB-07_salida_cnf_trim_cgr

Process Flow SAS: **Salida_CNF_Trim_CGR** — `ProcessFlowContainer-wkiL22TwhM21CAzc`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

Year = None  # &Year — nadie declaró su valor

faltantes = [n for n, v in {"Year": Year}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text

# Conexión a BD — editable acá; SASMIG_DB_URL (orquestador) tiene
# prioridad si está definida (SUPUESTO: verificar servidor y base
# antes de correr contra datos reales).
config_db = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=PLATDAT,1433;"
    "DATABASE=GOBGENER;"
    "Authentication=ActiveDirectoryIntegrated;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "MARS_Connection=Yes;"
)
engine = sqlalchemy.create_engine(
    os.environ.get("SASMIG_DB_URL", f"mssql+pyodbc:///?odbc_connect={config_db}"),
    pool_pre_ping=True,
    fast_executemany=True,
)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-07_salida_cnf_trim_cgr.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## 1- TCons Mixtas_M_CGR

Redistribuye la ejecución CGR del año en transacciones Mixtas (M), combinando la muestra estructurada con el resto expandido por ponderación, ajusta por la imputación de pandemia 2020 y produce los totales trimestrales agregados y desagregados por entidad

*confianza: medium · verificador: approve · SAS: PROC SQL: filtro por año + LEFT JOINs de redistribución con ponderación POND + UNION ALL + DELETE de ajuste + agregaciones finales*

In [ ]:
# ========= 1- TCons Mixtas_M_CGR =========
# /*FILTRA DATOS DE EJECUCION CGR DEL AÑO*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr_y"))
# Year interpolado como entero: un bind mataría la #tmp (sp_prepexec)
sql_ejec_cgr_y = f"""
SELECT *
INTO #ejec_cgr_y
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejec_cgr_y))

# DATA step DROP de columnas no usadas aguas abajo
work_conn.execute(text("ALTER TABLE #ejec_cgr_y DROP COLUMN PPTO_INICIAL, PPTO_ACTUAL, PERCIB_PAG, POR_PERCIB_PAG, N_ITEM, N_ASIGNACION, C_SUBASIGNACION, N_SUBASIGNACION"))


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MIXTAS (M) SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
# El faltante de SAS (T2.DEVENG<>.) tras el LEFT JOIN es NULL en SQL Server: se excluye con IS NOT NULL
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m"))
sql_ejec_muestra_mixtas_m = """
SELECT t2.SUBSECTOR, t2.MONEDA, t2.MES, t2.AÑO, t1.ENTIDAD AS C_ENTIDAD,
       t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO, t2.C_PROGRAMA, t2.N_PROGRAMA,
       t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
       SUM(t2.DEVENG/1000000.0*t1.POND) AS DEVENGO
INTO #ejec_muestra_mixtas_m
FROM GOBGENER.dbo.EST_MIXTAS_M t1
LEFT JOIN #ejec_cgr_y t2 ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
    AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24','33') AND t2.C_ITEM IN ('03','01','06','08','09') AND t1.ENTIDAD NOT IN (5601,10201)
    AND t2.DEVENG IS NOT NULL
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.MES, t2.AÑO, t1.ENTIDAD,
         t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO, t2.C_PROGRAMA, t2.N_PROGRAMA,
         t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
         t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
work_conn.execute(text(sql_ejec_muestra_mixtas_m))
ejec_muestra_mixtas_m = pd.read_sql(text("SELECT * FROM #ejec_muestra_mixtas_m"), work_conn)
_log("ejec_muestra_mixtas_m", ejec_muestra_mixtas_m)


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MIXTAS (M) A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
# T2.DEVENG<>. tras el LEFT JOIN -> IS NOT NULL en SQL Server
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mixtas_m"))
sql_cruce_ejec_exp_mixtas_m = """
SELECT t2.*,
       t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.Clasif_Tcons
INTO #cruce_ejec_exp_mixtas_m
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_cgr_y t2 ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
    AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0 AND t2.C_ENTIDAD NOT IN (5601,10201) AND t1.Clasif_Tcons = 'M' AND t2.DEVENG IS NOT NULL
"""
work_conn.execute(text(sql_cruce_ejec_exp_mixtas_m))
cruce_ejec_exp_mixtas_m = pd.read_sql(text("SELECT * FROM #cruce_ejec_exp_mixtas_m"), work_conn)
_log("cruce_ejec_exp_mixtas_m", cruce_ejec_exp_mixtas_m)


In [ ]:
# /* CREA TABLA DE PROM_MES (dependencia previa del flujo) */
# PROM_MES es un WORK que este nodo consume; se materializa en sesión si aún no existe como #tmp
prom_mes.to_sql("#prom_mes", work_conn, if_exists="replace", index=False)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MIXTAS (M), PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim"))
sql_ejec_exp_mixtas_m_prelim = """
SELECT t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD,
       t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO, t2.C_PROGRAMA, t2.N_PROGRAMA,
       t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
       t2.DEVENG, t2.Clasif_Tcons,
       t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
       SUM(t2.DEVENG) AS DATO_OR,
       SUM(t2.DEVENG/1000000.0*t1.POND*t3.DATO) AS DEVENGO
INTO #ejec_exp_mixtas_m_prelim
FROM GOBGENER.dbo.EST_EXP_MIXTAS_M t1
LEFT JOIN #cruce_ejec_exp_mixtas_m t2 ON t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM AND t2.C_ASIGNACION = t1.C_ASIGNACION
    AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.MES = t1.MES AND t2.C_ENTIDAD = t1.ENTIDAD
LEFT JOIN #prom_mes t3 ON t2.MONEDA = t3.MONEDA AND t2.MES = t3.MES AND t2.AÑO = t3.AÑO
WHERE t2.DEVENG IS NOT NULL
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD,
         t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO, t2.C_PROGRAMA, t2.N_PROGRAMA,
         t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
         t2.DEVENG, t2.Clasif_Tcons, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
work_conn.execute(text(sql_ejec_exp_mixtas_m_prelim))


In [ ]:
# DATA step: EJEC_EXP_MIXTAS_M = EJEC_EXP_MIXTAS_M_PRELIM (DROP=DEVENG DATO_OR Clasif_Tcons)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m"))
sql_ejec_exp_mixtas_m = """
SELECT SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD,
       C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA,
       C_TIPO_CUENTA, C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
       C_SCN, C_CAGENTE, C_CUP, DEVENGO
INTO #ejec_exp_mixtas_m
FROM #ejec_exp_mixtas_m_prelim
"""
work_conn.execute(text(sql_ejec_exp_mixtas_m))
ejec_exp_mixtas_m_prelim = pd.read_sql(text("SELECT * FROM #ejec_exp_mixtas_m_prelim"), work_conn)
_log("ejec_exp_mixtas_m_prelim", ejec_exp_mixtas_m_prelim)


In [ ]:
# /* Crea tabla con la ejecución total de transacciones Mixtas (M), Muestra + Resto */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_total"))
sql_ejec_mixtas_m_total = """
SELECT SUBSECTOR, MONEDA, MES, AÑO, C_ENTIDAD,
       C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA,
       C_TIPO_CUENTA, C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
       C_SCN, C_CAGENTE, C_CUP, DEVENGO
INTO #ejec_mixtas_m_total
FROM #ejec_muestra_mixtas_m
UNION ALL
SELECT SUBSECTOR, MONEDA, MES, AÑO, C_ENTIDAD,
       C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO, C_PROGRAMA, N_PROGRAMA,
       C_TIPO_CUENTA, C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
       C_SCN, C_CAGENTE, C_CUP, DEVENGO
FROM #ejec_exp_mixtas_m
"""
work_conn.execute(text(sql_ejec_mixtas_m_total))


In [ ]:
# /* Ajuste por imputación de D63 en pandemia (2020) */
# Hard-coded según el SAS original: entidad 911, cuenta 24, item 03, asignacion 002, analitico 03002000
res = work_conn.execute(text(
    "DELETE FROM #ejec_mixtas_m_total "
    "WHERE C_ENTIDAD = 911 AND C_CUENTA = '24' AND C_ITEM = '03' AND C_ASIGNACION = '002' "
    "AND C_ANALITICO IN ('03002000') AND AÑO = 2020 AND MES > 4"
))
_log("DELETE #ejec_mixtas_m_total (ajuste pandemia)", res.rowcount)
ejec_mixtas_m_total = pd.read_sql(text("SELECT * FROM #ejec_mixtas_m_total"), work_conn)
_log("ejec_mixtas_m_total", ejec_mixtas_m_total)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CNF trimestral */
# TRIM es función pura de MES (CASE WHEN sobre columna del GROUP BY): se calcula sobre el agregado
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mixt"))
sql_ejec_e_mixt = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
       SUM(t1.DEVENGO) AS Dato, t3.ACTIVIDAD, t3.NIVEL
INTO #ejec_e_mixt
FROM #ejec_mixtas_m_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
         CASE WHEN t1.MES IN (1,2,3) THEN 1
              WHEN t1.MES IN (4,5,6) THEN 2
              WHEN t1.MES IN (7,8,9) THEN 3
              ELSE 4 END,
         t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.N_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE,
         t3.ACTIVIDAD, t3.NIVEL, t1.C_CUP
"""
work_conn.execute(text(sql_ejec_e_mixt))
ejec_e_mixt = pd.read_sql(text("SELECT * FROM #ejec_e_mixt"), work_conn)
_log("ejec_e_mixt", ejec_e_mixt)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para ANALISIS DESAGREGADO CNF trimestral */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mixt_aper"))
sql_ejec_e_mixt_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE,
       SUM(t1.DEVENGO) AS Dato, t3.ACTIVIDAD, t3.NIVEL
INTO #ejec_e_mixt_aper
FROM #ejec_mixtas_m_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
         CASE WHEN t1.MES IN (1,2,3) THEN 1
              WHEN t1.MES IN (4,5,6) THEN 2
              WHEN t1.MES IN (7,8,9) THEN 3
              ELSE 4 END,
         t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
         t1.C_SCN, t1.C_CAGENTE, t3.ACTIVIDAD, t3.NIVEL
"""
work_conn.execute(text(sql_ejec_e_mixt_aper))
ejec_e_mixt_aper = pd.read_sql(text("SELECT * FROM #ejec_e_mixt_aper"), work_conn)
_log("ejec_e_mixt_aper", ejec_e_mixt_aper)


In [ ]:
# PROC SQL DROP TABLE de intermedias: limpieza de sesión (equivalente al DROP TABLE del SAS)
for _t in ["#ejec_muestra_mixtas_m", "#cruce_ejec_exp_mixtas_m", "#ejec_exp_mixtas_m_prelim", "#ejec_exp_mixtas_m", "#ejec_mixtas_m_total"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 2- TCons MGCG_CGR

Redistribuye las transacciones MGCG (cuenta 24, item 03) según estructura de muestra y expande el resto según ponderadores y promedios mensuales, generando la ejecución total y sus salidas agregadas para CNF trimestral

*confianza: medium · verificador: approve · SAS: PROC SQL: LEFT JOINs con agregación GROUP BY, expansión de muestra y UNION vía concatenación de datasets*

In [ ]:
# ========= 2- TCons MGCG_CGR =========
# CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MGCG SEGUN ESTRUCTURA, PARA EJECUCION AÑO t
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mgcg"))
sql_ejec_muestra_mgcg = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t2.DEVENG/1000000*t1.POND) AS DEVENGO
INTO #ejec_muestra_mgcg
FROM GOBGENER.dbo.EST_MGCG t1
LEFT JOIN #ejec_cgr_y t2 ON (t2.C_ENTIDAD=t1.ENTIDAD) AND (t2.C_ASIGNACION=t1.C_ASIGNACION) AND
    (t2.C_ANALITICO=t1.C_ANALITICO) AND (t2.C_CUENTA=t1.C_CUENTA) AND (t2.C_ITEM=t1.C_ITEM) AND (t2.MES=t1.MES)
WHERE t2.C_CUENTA IN ('24') AND t2.C_ITEM IN ('03') AND t1.ENTIDAD NOT IN (5601,10201) AND (T2.DEVENG IS NOT NULL)
GROUP BY T2.SUBSECTOR,T2.MONEDA,t2.MES,
    t2.AÑO,t1.ENTIDAD,T2.C_PARTIDA,T2.N_PARTIDA,T2.C_CAPITULO,T2.N_CAPITULO,T2.C_PROGRAMA,T2.N_PROGRAMA,
    T2.C_TIPO_CUENTA,t2.C_CUENTA,t2.N_CUENTA,t2.C_ITEM,t2.C_ASIGNACION,t2.C_ANALITICO,t2.N_ANALITICO,t1.C_SCN,
    t1.C_CAGENTE,t2.DEVENG
"""
work_conn.execute(text(sql_ejec_muestra_mgcg))


In [ ]:
# DATA EJEC_MUESTRA_MGCG; SET EJEC_MUESTRA_MGCG (DROP=DEVENG); RUN;
# El DROP de una sola columna se resuelve trayendo el resultado a pandas,
# quitando DEVENG y volviendo a materializar la #tmp con ese esquema
ejec_muestra_mgcg = pd.read_sql(text("SELECT * FROM #ejec_muestra_mgcg"), work_conn)
ejec_muestra_mgcg = ejec_muestra_mgcg.drop(columns=["DEVENG"])
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mgcg"))
ejec_muestra_mgcg.to_sql("#ejec_muestra_mgcg", work_conn, if_exists="replace", index=False)
_log("ejec_muestra_mgcg", ejec_muestra_mgcg)


In [ ]:
# CREA TABLA DE ANALITICOS DE MGCG A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mgcg"))
sql_cruce_ejec_exp_mgcg = """
SELECT t2.*,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.Clasif_Tcons
INTO #cruce_ejec_exp_mgcg
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_cgr_y t2 ON (t2.C_ENTIDAD=t1.ENTIDAD) AND (t2.C_ASIGNACION=t1.C_ASIGNACION) AND
    (t2.C_ANALITICO=t1.C_ANALITICO) AND (t2.C_CUENTA=t1.C_CUENTA) AND (t2.C_ITEM=t1.C_ITEM)
WHERE t1.M_MUESTRA=0 AND t1.Clasif_Tcons='MGCG' AND t2.C_ENTIDAD NOT IN (5601,10201) AND (t2.DEVENG IS NOT NULL)
"""
work_conn.execute(text(sql_cruce_ejec_exp_mgcg))
cruce_ejec_exp_mgcg = pd.read_sql(text("SELECT * FROM #cruce_ejec_exp_mgcg"), work_conn)
_log("cruce_ejec_exp_mgcg", cruce_ejec_exp_mgcg)


In [ ]:
# CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MGCG,
# PARA EJECUCION AÑO t
# NOTA: PROM_MES tiene clave compuesta año/mes/moneda sin validación de existencia completa;
# un NULL en t3.DATO (PROM_MES) genera DEVENGO nulo (riesgo señalado por el analista)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgcg_prelim"))
sql_ejec_exp_mgcg_prelim = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t2.DEVENG) AS DATO_OR,
       SUM(t2.DEVENG/1000000*t1.POND*t3.DATO) AS DEVENGO
INTO #ejec_exp_mgcg_prelim
FROM GOBGENER.dbo.EST_EXP_MGCG t1
LEFT JOIN #cruce_ejec_exp_mgcg t2 ON (t2.C_CUENTA=t1.C_CUENTA) AND (t2.C_ITEM=t1.C_ITEM) AND (t2.MES=t1.MES)
LEFT JOIN #prom_mes t3 ON (t2.MONEDA = t3.MONEDA) AND (t2.MES = t3.MES) AND (t2.AÑO = t3.AÑO)
WHERE (t2.DEVENG IS NOT NULL)
GROUP BY t2.SUBSECTOR,t2.MONEDA, t2.AÑO,
    t2.MES,
    t2.C_ENTIDAD,
    t2.C_PARTIDA,
    t2.N_PARTIDA,
    t2.C_CAPITULO,
    t2.N_CAPITULO,
    t2.C_PROGRAMA,
    t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA,
    t2.C_CUENTA,
    t2.N_CUENTA,
    t2.C_ITEM,
    t2.C_ASIGNACION,
    t2.C_ANALITICO,
    t2.N_ANALITICO,
    t2.DEVENG,
    t2.Clasif_Tcons,t1.C_SCN,
    t1.C_CAGENTE
"""
work_conn.execute(text(sql_ejec_exp_mgcg_prelim))
ejec_exp_mgcg_prelim = pd.read_sql(text("SELECT * FROM #ejec_exp_mgcg_prelim"), work_conn)
_log("ejec_exp_mgcg_prelim", ejec_exp_mgcg_prelim)


In [ ]:
# DATA EJEC_EXP_MGCG; SET EJEC_EXP_MGCG_prelim (DROP=DEVENG DATO_OR Clasif_Tcons); RUN;
ejec_exp_mgcg = ejec_exp_mgcg_prelim.drop(columns=["DEVENG", "DATO_OR", "Clasif_Tcons"])
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgcg"))
ejec_exp_mgcg.to_sql("#ejec_exp_mgcg", work_conn, if_exists="replace", index=False)
_log("ejec_exp_mgcg", ejec_exp_mgcg)


In [ ]:
# Crea tabla con la ejecucion total de transacciones MGCG, Muestra + Resto
ejec_mgcg_total = pd.concat([ejec_muestra_mgcg, ejec_exp_mgcg], ignore_index=True)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mgcg_total"))
ejec_mgcg_total.to_sql("#ejec_mgcg_total", work_conn, if_exists="replace", index=False)
_log("ejec_mgcg_total", ejec_mgcg_total)


In [ ]:
# SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGCG para CNF trimestral
sql_ejec_e_mgcg = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t1.DEVENGO) AS DEVENGO,t3.ACTIVIDAD,t3.NIVEL
FROM #ejec_mgcg_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON (t1.C_ENTIDAD=t3.ENTIDAD)
GROUP BY t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END),
    t1.MES,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_SCN,
    t1.C_CAGENTE,t3.ACTIVIDAD,t3.NIVEL
"""
ejec_e_mgcg = pd.read_sql(text(sql_ejec_e_mgcg), work_conn)
_log("ejec_e_mgcg", ejec_e_mgcg)


In [ ]:
# SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGCG para ANALISIS DESAGREGADO CNF trimestral
sql_ejec_e_mgcg_aper = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t1.DEVENGO) AS DEVENGO,t3.ACTIVIDAD,t3.NIVEL
FROM #ejec_mgcg_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON (t1.C_ENTIDAD=t3.ENTIDAD)
GROUP BY t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END),
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    t1.C_SCN,
    t1.C_CAGENTE,t3.ACTIVIDAD,t3.NIVEL
"""
ejec_e_mgcg_aper = pd.read_sql(text(sql_ejec_e_mgcg_aper), work_conn)
_log("ejec_e_mgcg_aper", ejec_e_mgcg_aper)


In [ ]:
# DROP TABLE EJEC_MUESTRA_MGCG,CRUCE_EJEC_EXP_MGCG,EJEC_EXP_MGCG_prelim,EJEC_EXP_MGCG,EJEC_MGCG_TOTAL;
for _t in ['#ejec_muestra_mgcg', '#cruce_ejec_exp_mgcg', '#ejec_exp_mgcg_prelim', '#ejec_exp_mgcg', '#ejec_mgcg_total']:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 3- TCons MM_CGR

Redistribuye la muestra y expande el resto de transacciones MM según estructura de ponderadores (POND) y factor de promedio mensual (PROM_MES), consolidando la ejecución total MM del año por trimestre, cuenta y apertura institucional

*confianza: medium · verificador: approve · SAS: PROC SQL: LEFT JOIN + agregación de muestra, expansión de analíticos con ponderador y promedio mensual, concatenación y agregación final por trimestre/apertura*

In [ ]:
# ========= 3- TCons MM_CGR =========
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mm"))
# CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MM SEGUN ESTRUCTURA, PARA EJECUCION AÑO t
# T2.DEVENG<>. tras LEFT JOIN -> IS NOT NULL (missing SAS = NULL SQL Server)
sql_ejec_muestra_mm = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t2.DEVENG/1000000*t1.POND) AS DEVENGO
INTO #ejec_muestra_mm
FROM GOBGENER.dbo.EST_MM t1
LEFT JOIN #ejec_cgr_y t2 ON (t2.C_ENTIDAD=t1.ENTIDAD) AND (t2.C_ASIGNACION=t1.C_ASIGNACION) AND
    (t2.C_ANALITICO=t1.C_ANALITICO) AND (t2.C_CUENTA=t1.C_CUENTA) AND (t2.C_ITEM=t1.C_ITEM) AND (t2.MES=t1.MES)
WHERE t2.C_CUENTA IN ('24','33') AND t2.C_ITEM IN ('01','03','08','09') AND t2.DEVENG IS NOT NULL AND t1.ENTIDAD NOT IN (5601,10201)
GROUP BY t2.SUBSECTOR,t2.MONEDA,t2.MES,
         t2.AÑO,t1.ENTIDAD,t2.C_PARTIDA,t2.N_PARTIDA,t2.C_CAPITULO,t2.N_CAPITULO,t2.C_PROGRAMA,t2.N_PROGRAMA,
         t2.C_TIPO_CUENTA,t2.C_CUENTA,t2.N_CUENTA,t2.C_ITEM,t2.C_ASIGNACION,t2.C_ANALITICO,t2.N_ANALITICO,t1.C_SCN,
         t1.C_CAGENTE,t2.DEVENG
"""
work_conn.execute(text(sql_ejec_muestra_mm))


In [ ]:
# DATA EJEC_MUESTRA_MM; SET EJEC_MUESTRA_MM (DROP=DEVENG) -> descarta la columna DEVENG (queda DEVENGO agregado)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mm_f"))
work_conn.execute(text("SELECT * INTO #ejec_muestra_mm_f FROM #ejec_muestra_mm"))
work_conn.execute(text("ALTER TABLE #ejec_muestra_mm_f DROP COLUMN DEVENG"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mm"))
work_conn.execute(text("EXEC sp_rename '#ejec_muestra_mm_f', '#ejec_muestra_mm'"))
ajec_muestra_mm_count = pd.read_sql(text("SELECT COUNT(*) AS n FROM #ejec_muestra_mm"), work_conn)
_log("ejec_muestra_mm", ajec_muestra_mm_count)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mm"))
# CREA TABLA DE ANALITICOS DE MM A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t
sql_cruce_ejec_exp_mm = """
SELECT t2.*,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.ENTIDAD,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.Clasif_Tcons
INTO #cruce_ejec_exp_mm
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_cgr_y t2 ON (t2.C_ENTIDAD=t1.ENTIDAD) AND (t2.C_ASIGNACION=t1.C_ASIGNACION) AND
    (t2.C_ANALITICO=t1.C_ANALITICO) AND (t2.C_CUENTA=t1.C_CUENTA) AND (t2.C_ITEM=t1.C_ITEM)
WHERE t1.M_MUESTRA=0 AND t1.Clasif_Tcons='MM' AND t1.ENTIDAD NOT IN (5601,10201) AND t2.DEVENG IS NOT NULL
"""
work_conn.execute(text(sql_cruce_ejec_exp_mm))
cruce_ejec_exp_mm = pd.read_sql(text("SELECT * FROM #cruce_ejec_exp_mm"), work_conn)
_log("cruce_ejec_exp_mm", cruce_ejec_exp_mm)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mm_prelim"))
# CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MM, PARA EJECUCION AÑO t
sql_ejec_exp_mm_prelim = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t2.DEVENG) AS DATO_OR,
       SUM(t2.DEVENG/1000000*t1.POND*t3.DATO) AS DEVENGO
INTO #ejec_exp_mm_prelim
FROM GOBGENER.dbo.EST_EXP_MM t1
LEFT JOIN #cruce_ejec_exp_mm t2 ON (t2.C_CUENTA=t1.C_CUENTA) AND (t2.C_ITEM=t1.C_ITEM) AND (t2.MES=t1.MES) AND (t1.ENTIDAD=t2.C_ENTIDAD)
    AND (t1.C_ASIGNACION=t2.C_ASIGNACION) AND (t1.C_ANALITICO=t2.C_ANALITICO)
LEFT JOIN #prom_mes t3 ON (t2.MONEDA = t3.MONEDA) AND (t2.MES = t3.MES) AND (t2.AÑO = t3.AÑO)
WHERE t2.DEVENG IS NOT NULL
GROUP BY t2.SUBSECTOR,t2.MONEDA, t2.AÑO,
         t2.MES,
         t2.C_ENTIDAD,
         t2.C_PARTIDA,
         t2.N_PARTIDA,
         t2.C_CAPITULO,
         t2.N_CAPITULO,
         t2.C_PROGRAMA,
         t2.N_PROGRAMA,
         t2.C_TIPO_CUENTA,
         t2.C_CUENTA,
         t2.N_CUENTA,
         t2.C_ITEM,
         t2.C_ASIGNACION,
         t2.C_ANALITICO,
         t2.N_ANALITICO,
         t2.DEVENG,
         t2.Clasif_Tcons,t1.C_SCN,
         t1.C_CAGENTE
"""
work_conn.execute(text(sql_ejec_exp_mm_prelim))
ejec_exp_mm_prelim = pd.read_sql(text("SELECT * FROM #ejec_exp_mm_prelim"), work_conn)
_log("ejec_exp_mm_prelim", ejec_exp_mm_prelim)


In [ ]:
# DATA EJEC_EXP_MM; SET EJEC_EXP_MM_prelim (DROP=DEVENG DATO_OR Clasif_Tcons)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mm"))
work_conn.execute(text("SELECT * INTO #ejec_exp_mm FROM #ejec_exp_mm_prelim"))
work_conn.execute(text("ALTER TABLE #ejec_exp_mm DROP COLUMN DEVENG, DATO_OR, Clasif_Tcons"))
ejec_exp_mm = pd.read_sql(text("SELECT * FROM #ejec_exp_mm"), work_conn)
_log("ejec_exp_mm", ejec_exp_mm)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mm_total"))
# Crea tabla con la ejecución total de transacciones MGG, Muestra + Resto
work_conn.execute(text("SELECT * INTO #ejec_mm_total FROM #ejec_exp_mm"))
work_conn.execute(text("INSERT INTO #ejec_mm_total SELECT * FROM #ejec_muestra_mm"))
ejec_mm_total = pd.read_sql(text("SELECT * FROM #ejec_mm_total"), work_conn)
_log("ejec_mm_total", ejec_mm_total)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mm"))
# SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGG para CNF trimestral
# TRIM es función pura de MES (GROUP BY lo incluye vía CASE en el select final)
sql_ejec_e_mm = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t1.DEVENGO) AS DEVENGO,t3.ACTIVIDAD,t3.NIVEL
INTO #ejec_e_mm
FROM #ejec_mm_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON (t1.C_ENTIDAD=t3.ENTIDAD)
GROUP BY t1.SUBSECTOR,
         t1.MONEDA,
         t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END),
         t1.MES,
         t1.C_TIPO_CUENTA,
         t1.C_CUENTA,
         t1.C_ITEM,
         t1.C_SCN,
         t1.C_CAGENTE,t3.ACTIVIDAD,t3.NIVEL
"""
work_conn.execute(text(sql_ejec_e_mm))
ajec_e_mm = pd.read_sql(text("SELECT * FROM #ejec_e_mm"), work_conn)
_log("ejec_e_mm", ajec_e_mm)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mm_aper"))
# SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGG para ANALISIS DESAGREGADO CNF trim
sql_ejec_e_mm_aper = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t1.DEVENGO) AS DEVENGO,t3.ACTIVIDAD,t3.NIVEL
INTO #ejec_e_mm_aper
FROM #ejec_mm_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON (t1.C_ENTIDAD=t3.ENTIDAD)
GROUP BY t1.SUBSECTOR,
         t1.MONEDA,
         t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END),
         t1.MES,
         t1.C_ENTIDAD,
         t1.C_PARTIDA,
         t1.C_CAPITULO,
         t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA,
         t1.C_CUENTA,
         t1.C_ITEM,
         t1.C_ASIGNACION,
         t1.C_ANALITICO,
         t1.N_ANALITICO,
         t1.C_SCN,
         t1.C_CAGENTE,t3.ACTIVIDAD,t3.NIVEL
"""
work_conn.execute(text(sql_ejec_e_mm_aper))
ajec_e_mm_aper = pd.read_sql(text("SELECT * FROM #ejec_e_mm_aper"), work_conn)
_log("ejec_e_mm_aper", ajec_e_mm_aper)


In [ ]:
# DROP TABLE EJEC_MUESTRA_MM,CRUCE_EJEC_EXP_MM,EJEC_EXP_MM_prelim,EJEC_EXP_MM,EJEC_MM_TOTAL
for _t in ["#ejec_muestra_mm", "#cruce_ejec_exp_mm", "#ejec_exp_mm_prelim", "#ejec_exp_mm", "#ejec_mm_total"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 4- TCons MGG_CGR

Redistribuye la muestra de transacciones MGG según estructura de ponderadores, expande los analíticos fuera de muestra usando el promedio mensual, consolida la ejecución total MGG (muestra + resto) y genera las salidas agregadas para CNF trimestral, tanto a nivel general como desagregado por entidad/analítico

*confianza: medium · verificador: approve · SAS: PROC SQL: LEFT JOIN + agregación para redistribución de muestra y expansión de analíticos MGG, UNION ALL de totales y salidas agregadas por trimestre*

In [ ]:
# ========= 4- TCons MGG_CGR =========
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mgg"))
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MGG SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
sql_ejec_muestra_mgg = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t2.DEVENG/1000000*t1.POND) AS DEVENGO
INTO #ejec_muestra_mgg
FROM GOBGENER.dbo.EST_MGG t1
LEFT JOIN #ejec_cgr_y t2 ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
    AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24') AND t2.C_ITEM IN ('01','03','08','09') AND t1.ENTIDAD NOT IN (5601,10201)
    AND (t2.DEVENG IS NOT NULL)
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.MES,
    t2.AÑO, t1.ENTIDAD, t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO, t2.C_PROGRAMA, t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO, t1.C_SCN,
    t1.C_CAGENTE, t2.DEVENG
"""
work_conn.execute(text(sql_ejec_muestra_mgg))


In [ ]:
# DATA EJEC_MUESTRA_MGG; SET EJEC_MUESTRA_MGG (DROP=DEVENG); -> quita la columna auxiliar DEVENG (queda DEVENGO)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mgg_2"))
work_conn.execute(text("""
SELECT * INTO #ejec_muestra_mgg_2
FROM (SELECT * FROM #ejec_muestra_mgg) x
"""))
work_conn.execute(text("ALTER TABLE #ejec_muestra_mgg_2 DROP COLUMN DEVENG"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mgg"))
work_conn.execute(text("EXEC sp_rename '#ejec_muestra_mgg_2', '#ejec_muestra_mgg'"))
ejec_muestra_mgg = pd.read_sql(text("SELECT * FROM #ejec_muestra_mgg"), work_conn)
_log("ejec_muestra_mgg", ejec_muestra_mgg)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mgg"))
# /* CREA TABLA DE ANALITICOS DE MGG A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
sql_cruce_ejec_exp_mgg = """
SELECT t2.*,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.Clasif_Tcons
INTO #cruce_ejec_exp_mgg
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_cgr_y t2 ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
    AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0 AND t1.Clasif_Tcons = 'MGG' AND t2.C_ENTIDAD NOT IN (5601,10201)
    AND (t2.DEVENG IS NOT NULL)
"""
work_conn.execute(text(sql_cruce_ejec_exp_mgg))
cruce_ejec_exp_mgg = pd.read_sql(text("SELECT * FROM #cruce_ejec_exp_mgg"), work_conn)
_log("cruce_ejec_exp_mgg", cruce_ejec_exp_mgg)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgg_preliminar"))
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MGG, PARA EJECUCION AÑO t */
# t2.'AÑO'n / t3.'AÑO'n del SAS son la misma columna AÑO en ambas tablas (quoted name por el caracter especial)
sql_ejec_exp_mgg_preliminar = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t2.DEVENG) AS DATO_OR,
       SUM(t2.DEVENG/1000000*t1.POND*t3.DATO) AS DEVENGO
INTO #ejec_exp_mgg_preliminar
FROM GOBGENER.dbo.EST_EXP_MGG t1
LEFT JOIN #cruce_ejec_exp_mgg t2 ON t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM AND t2.MES = t1.MES
LEFT JOIN #prom_mes t3 ON t2.MONEDA = t3.MONEDA AND t2.MES = t3.MES AND t2.AÑO = t3.AÑO
WHERE t2.DEVENG IS NOT NULL
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.AÑO,
    t2.MES, t2.C_ENTIDAD, t2.C_PARTIDA, t2.N_PARTIDA, t2.C_CAPITULO, t2.N_CAPITULO, t2.C_PROGRAMA, t2.N_PROGRAMA,
    t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.N_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
    t2.DEVENG, t2.Clasif_Tcons, t1.C_SCN, t1.C_CAGENTE
"""
work_conn.execute(text(sql_ejec_exp_mgg_preliminar))
ejec_exp_mgg_preliminar = pd.read_sql(text("SELECT * FROM #ejec_exp_mgg_preliminar"), work_conn)
_log("ejec_exp_mgg_preliminar", ejec_exp_mgg_preliminar)


In [ ]:
# DATA EJEC_EXP_MGG; SET EJEC_EXP_MGG_preliminar (DROP=DEVENG DATO_OR Clasif_Tcons);
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mgg"))
work_conn.execute(text("""
SELECT * INTO #ejec_exp_mgg
FROM (SELECT * FROM #ejec_exp_mgg_preliminar) x
"""))
work_conn.execute(text("ALTER TABLE #ejec_exp_mgg DROP COLUMN DEVENG, DATO_OR, Clasif_Tcons"))
ejec_exp_mgg = pd.read_sql(text("SELECT * FROM #ejec_exp_mgg"), work_conn)
_log("ejec_exp_mgg", ejec_exp_mgg)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mgg_total"))
# /* Crea tabla con la ejecución total de transacciones MGG, Muestra + Resto */
work_conn.execute(text("""
SELECT * INTO #ejec_mgg_total FROM #ejec_muestra_mgg
UNION ALL
SELECT * FROM #ejec_exp_mgg
"""))
ejec_mgg_total = pd.read_sql(text("SELECT * FROM #ejec_mgg_total"), work_conn)
_log("ejec_mgg_total", ejec_mgg_total)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mgg"))
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGG para CNF trimestral */
sql_ejec_e_mgg = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t1.DEVENGO) AS DEVENGO,
       t3.ACTIVIDAD,
       t3.NIVEL
INTO #ejec_e_mgg
FROM #ejec_mgg_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1,2,3) THEN 1
         WHEN t1.MES IN (4,5,6) THEN 2
         WHEN t1.MES IN (7,8,9) THEN 3
         ELSE 4 END,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.N_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t3.ACTIVIDAD, t3.NIVEL
"""
work_conn.execute(text(sql_ejec_e_mgg))
ejec_e_mgg = pd.read_sql(text("SELECT * FROM #ejec_e_mgg"), work_conn)
_log("ejec_e_mgg", ejec_e_mgg)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_e_mgg_aper"))
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MGG para ANALISIS DESAGREGADO CNF trim */
sql_ejec_e_mgg_aper = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1
            WHEN t1.MES IN (4,5,6) THEN 2
            WHEN t1.MES IN (7,8,9) THEN 3
            ELSE 4 END AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       SUM(t1.DEVENGO) AS DEVENGO,
       t3.ACTIVIDAD,
       t3.NIVEL
INTO #ejec_e_mgg_aper
FROM #ejec_mgg_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1,2,3) THEN 1
         WHEN t1.MES IN (4,5,6) THEN 2
         WHEN t1.MES IN (7,8,9) THEN 3
         ELSE 4 END,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t3.ACTIVIDAD, t3.NIVEL
"""
work_conn.execute(text(sql_ejec_e_mgg_aper))
ejec_e_mgg_aper = pd.read_sql(text("SELECT * FROM #ejec_e_mgg_aper"), work_conn)
_log("ejec_e_mgg_aper", ejec_e_mgg_aper)


## SALIDA_CNF_trim

Consolida la ejecución no financiera del Gobierno General por trimestre a partir de la base CGR y las 4 tablas TCons de aperturas, aplicando ~15 reglas de reasignación de códigos de transacción y sector consolidado por año/entidad/cuenta, y separa la extracción de ingresos municipales del periodo

*confianza: medium · verificador: approve · SAS: PROC SQL CREATE TABLE/UPDATE/DELETE encadenados (4 bloques TCons + EJECUCION base) con reasignaciones hard-coded de C_SCN/C_CAGENTE/NIVEL, agregación final y extracción de ingresos municipales*

In [ ]:
# ========= SALIDA_CNF_trim =========
# /*Adjunta imputación por ajuste en IFE LABORAL 2023 */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion"))
sql_ejecucion = f"""
SELECT *
INTO #ejecucion
FROM (
    SELECT * FROM #ejec_cgr_y
    UNION ALL
    SELECT * FROM GOBGENER.dbo.AJUSTE_IFE_LABORAL
) u
WHERE AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejecucion))


In [ ]:
# /*Analiticos predefinidos */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim"))
sql_ejecucion_a_prelim = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_prelim
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
  ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
  ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_a_prelim))


In [ ]:
# DELETE analíticos cuenta 32 predefinidos que quedan fuera del bloque A
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"
))
_log("DELETE #ejecucion_a_prelim (cuenta 32)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se reasigna de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim WHERE C_ENTIDAD = 3801 AND C_CUENTA = '06' AND C_ITEM = '01' "
    "AND C_ASIGNACION = '000' AND C_ANALITICO IN ('01000000')"
))
_log("DELETE #ejecucion_a_prelim (ajuste coyuntura CORFO)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos: reasignación D39 2% constitucional 2008 para Transantiago */
res = work_conn.execute(text(
    "UPDATE #ejecucion_a_prelim SET C_CAGENTE = 'S11' "
    "WHERE C_ENTIDAD = 19703 AND C_CUENTA = '26' AND C_ITEM = '03' AND C_ASIGNACION = '000' "
    "AND C_ANALITICO IN ('03000000') AND MONEDA IN ('P') AND AÑO = 2008 AND MES IN (10,11,12)"
))
_log("UPDATE #ejecucion_a_prelim C_CAGENTE Transantiago", res.rowcount)

res = work_conn.execute(text(
    "UPDATE #ejecucion_a_prelim SET C_SCN = 'D39' "
    "WHERE C_ENTIDAD = 19703 AND C_CUENTA = '26' AND C_ITEM = '03' AND C_ASIGNACION = '000' "
    "AND C_ANALITICO IN ('03000000') AND MONEDA IN ('P') AND AÑO = 2008 AND MES IN (10,11,12)"
))
_log("UPDATE #ejecucion_a_prelim C_SCN Transantiago", res.rowcount)

res = work_conn.execute(text(
    "UPDATE #ejecucion_a_prelim SET C_TRANSACCION = '4139' "
    "WHERE C_ENTIDAD = 19703 AND C_CUENTA = '26' AND C_ITEM = '03' AND C_ASIGNACION = '000' "
    "AND C_ANALITICO IN ('03000000') AND MONEDA IN ('P') AND AÑO = 2008 AND MES IN (10,11,12)"
))
_log("UPDATE #ejecucion_a_prelim C_TRANSACCION Transantiago", res.rowcount)


In [ ]:
# /*Analiticos predefinidos: AJUSTE COBRE BRUTO (Financiamiento Capacidad Estrategicas Defensa
# Nacional) - reasignacion entre D75 y D42 */
res = work_conn.execute(text(
    "UPDATE #ejecucion_a_prelim SET C_SCN = 'D42' "
    "WHERE C_ENTIDAD = 19701 AND C_CUENTA = '08' AND C_ITEM = '99' AND C_ASIGNACION = '999' "
    "AND SUBSTRING(C_ANALITICO, 1, 5) = '99999' AND MONEDA IN ('D') AND AÑO > 2019"
))
_log("UPDATE #ejecucion_a_prelim C_SCN cobre bruto", res.rowcount)

res = work_conn.execute(text(
    "UPDATE #ejecucion_a_prelim SET C_TRANSACCION = '42222' "
    "WHERE C_ENTIDAD = 19701 AND C_CUENTA = '08' AND C_ITEM = '99' AND C_ASIGNACION = '999' "
    "AND SUBSTRING(C_ANALITICO, 1, 5) = '99999' AND MONEDA IN ('D') AND AÑO > 2019"
))
_log("UPDATE #ejecucion_a_prelim C_TRANSACCION cobre bruto", res.rowcount)

res = work_conn.execute(text(
    "UPDATE #ejecucion_a_prelim SET N_CUENTA = 'Cobre bruto' "
    "WHERE C_ENTIDAD = 19701 AND C_CUENTA = '08' AND C_ITEM = '99' AND C_ASIGNACION = '999' "
    "AND SUBSTRING(C_ANALITICO, 1, 5) = '99999' AND MONEDA IN ('D') AND AÑO > 2019"
))
_log("UPDATE #ejecucion_a_prelim N_CUENTA cobre bruto", res.rowcount)


In [ ]:
# /* Se eliminan Intereses en moneda dólar, no se consideran en indicador CR18 */
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim WHERE C_SCN = 'D41' AND MONEDA = 'D'"
))
_log("DELETE #ejecucion_a_prelim (intereses USD)", res.rowcount)

# /* Se eliminan dividendos TESORO, debido a que se toma dato de sectorialistas */
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim WHERE C_SCN IN ('D42','D421') AND C_PARTIDA = '50' AND C_CUENTA NOT IN ('08')"
))
_log("DELETE #ejecucion_a_prelim (dividendos Tesoro)", res.rowcount)


In [ ]:
# Agregación de EJECUCION_A: excluye SCN financieros y agrupa por dimensiones
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a"))
sql_ejecucion_a = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.C_ACT, t1.NIVEL, SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a
FROM #ejecucion_a_prelim t1
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
         t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
         t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_a))


In [ ]:
# /*Analiticos fijos cuenta 32 */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b"))
sql_ejecucion_b = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION,
       t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_b
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
  ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
     AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
  ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA = '32' AND t1.C_ANALITICO IN ('04001000','04002000') AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND (t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
       OR t2.C_SCN IS NULL)
"""
work_conn.execute(text(sql_ejecucion_b))


In [ ]:
# /*analíticos variables*/
# NOT IN sobre t2.C_SCN / t2.OBS tras LEFT JOIN: fila sin match (t2 NULL) queda
# excluida en SAS por el NOT IN sobre nulo -> se agrega IS NULL explícito solo
# donde el SAS también la habría dejado fuera del NOT IN por comparar con vacío ('')
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_prelim"))
sql_ejecucion_c_prelim = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION,
       t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_c_prelim
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
     AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
  ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND (t2.OBS NOT IN ('CR18_difcoy','CR18_difact') OR t2.OBS IS NULL)
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND (t2.C_SCN NOT IN ('TC') OR t2.C_SCN IS NULL)
  AND (t2.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
       OR t2.C_SCN IS NULL)
"""
work_conn.execute(text(sql_ejecucion_c_prelim))


In [ ]:
# /*Analiticos variables: reasignación contragente ayudas IPS */
# Bono Ayuda Familiar (18O)
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_prelim SET C_CAGENTE = 'S14' "
    "WHERE C_ENTIDAD = 24101 AND C_CUENTA = '24' AND C_ITEM = '01' AND C_ASIGNACION = '006' "
    "AND C_ANALITICO IN ('01006000') AND MONEDA IN ('P') AND AÑO = 2019 AND MES = 12"
))
_log("UPDATE #ejecucion_c_prelim Bono Ayuda Familiar", res.rowcount)

# Bono Covid
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_prelim SET C_CAGENTE = 'S14' "
    "WHERE C_ENTIDAD = 24101 AND C_CUENTA = '24' AND C_ITEM = '01' AND C_ASIGNACION = '006' "
    "AND C_ANALITICO IN ('01006000') AND MONEDA IN ('P') AND AÑO = 2020 AND MES = 4"
))
_log("UPDATE #ejecucion_c_prelim Bono Covid", res.rowcount)

# Reasignación nivel de gobierno para Programa Ampliado de Inmunizaciones
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_prelim SET NIVEL = 'MUNICIPAL' "
    "WHERE C_ENTIDAD IN (11105,22701,22750) AND C_CUENTA = '24' AND C_ITEM = '01' AND C_ASIGNACION = '004'"
))
_log("UPDATE #ejecucion_c_prelim PAI", res.rowcount)

# Reasignación nivel de D63 municipal
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_prelim SET NIVEL = 'MUNICIPAL' "
    "WHERE NIVEL = 'REGIONAL' AND C_SCN IN ('D63','D63-P2','D63-P') AND C_ITEM = '03' "
    "AND C_ASIGNACION IN ('001','003','006','011','013','015','016','017','018','020','021','100','101')"
))
_log("UPDATE #ejecucion_c_prelim D63 municipal", res.rowcount)

# Beca Especial Carbon (previo 2020)
res = work_conn.execute(text(
    "UPDATE #ejecucion_c_prelim SET C_CAGENTE = 'S14' "
    "WHERE C_ENTIDAD = 6403 AND C_CUENTA = '24' AND C_ITEM = '01' AND C_ASIGNACION = '270' "
    "AND C_ANALITICO IN ('01270000') AND MONEDA IN ('P') AND AÑO < 2020"
))
_log("UPDATE #ejecucion_c_prelim Beca Carbon C_CAGENTE", res.rowcount)

res = work_conn.execute(text(
    "UPDATE #ejecucion_c_prelim SET C_SCN = 'D75' "
    "WHERE C_ENTIDAD = 6403 AND C_CUENTA = '24' AND C_ITEM = '01' AND C_ASIGNACION = '270' "
    "AND C_ANALITICO IN ('01270000') AND MONEDA IN ('P') AND AÑO < 2020"
))
_log("UPDATE #ejecucion_c_prelim Beca Carbon C_SCN", res.rowcount)

res = work_conn.execute(text(
    "UPDATE #ejecucion_c_prelim SET C_TRANSACCION = '4345' "
    "WHERE C_ENTIDAD = 6403 AND C_CUENTA = '24' AND C_ITEM = '01' AND C_ASIGNACION = '270' "
    "AND C_ANALITICO IN ('01270000') AND MONEDA IN ('P') AND AÑO < 2020"
))
_log("UPDATE #ejecucion_c_prelim Beca Carbon C_TRANSACCION", res.rowcount)


In [ ]:
# Agregación de EJECUCION_C: excluye SCN financieros y agrupa
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c"))
sql_ejecucion_c = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.C_ACT, t1.NIVEL, SUM(t1.Dato) AS Dato
INTO #ejecucion_c
FROM #ejecucion_c_prelim t1
WHERE t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
         t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
         t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_c))


In [ ]:
# /* Transacciones diferenciadas para coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d"))
sql_ejecucion_d = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t2.C_ACTIVO AS C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION,
       t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_d
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
     AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
  ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difcoy') AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND (t2.C_SCN NOT IN ('TC') OR t2.C_SCN IS NULL)
  AND (t2.C_ACTIVO NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
       OR t2.C_ACTIVO IS NULL)
"""
work_conn.execute(text(sql_ejecucion_d))

# /* Reasignación nivel de D63 municipal */
res = work_conn.execute(text(
    "UPDATE #ejecucion_d SET NIVEL = 'MUNICIPAL' "
    "WHERE NIVEL = 'REGIONAL' AND C_ACTIVO IN ('D63') AND C_ITEM = '03'"
))
_log("UPDATE #ejecucion_d D63 municipal", res.rowcount)


In [ ]:
# /* Transacciones diferenciadas por actividad */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f"))
sql_ejecucion_f = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION,
       t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP, t2.C_CUP AS C_ACT, t4.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_f
FROM #ejecucion t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
     AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
  ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND (t2.C_SCN NOT IN ('TC') OR t2.C_SCN IS NULL)
  AND (t2.C_SCN NOT IN ('D73','D93','F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
       OR t2.C_SCN IS NULL)
"""
work_conn.execute(text(sql_ejecucion_f))


In [ ]:
# /* TCONS trimestrales desagregadas: une las 4 tablas TCons de aperturas (M-005: dependen
# de nodos previos) */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_e_prelim"))
sql_ejecucion_e_prelim = """
SELECT * INTO #ejecucion_e_prelim
FROM (
    SELECT * FROM #ejec_e_mixt_aper
    UNION ALL
    SELECT * FROM #ejec_e_mgcg_aper
    UNION ALL
    SELECT * FROM #ejec_e_mm_aper
    UNION ALL
    SELECT * FROM #ejec_e_mgg_aper
) u
"""
work_conn.execute(text(sql_ejecucion_e_prelim))

res = work_conn.execute(text("UPDATE #ejecucion_e_prelim SET C_ACTIVO = 'TC'"))
_log("UPDATE #ejecucion_e_prelim C_ACTIVO=TC", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_e"))
sql_ejecucion_e = """
SELECT *, ACTIVIDAD AS C_ACT
INTO #ejecucion_e
FROM #ejecucion_e_prelim
"""
work_conn.execute(text(sql_ejecucion_e))

res = work_conn.execute(text(
    "UPDATE #ejecucion_e SET C_ACT = 'AAPP' "
    "WHERE ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST')"
))
_log("UPDATE #ejecucion_e C_ACT=AAPP", res.rowcount)

res = work_conn.execute(text(
    "DELETE FROM #ejecucion_e WHERE C_SCN IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')"
))
_log("DELETE #ejecucion_e (SCN financieros)", res.rowcount)

# DATA step DROP=ACTIVIDAD: se descarta la columna de apoyo
work_conn.execute(text("ALTER TABLE #ejecucion_e DROP COLUMN ACTIVIDAD"))


In [ ]:
# /*UNE TABLAS DESAGREGADAS*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_trim_prev"))
sql_ejecucion_cnf_trim_prev = """
SELECT * INTO #ejecucion_cnf_trim_prev
FROM (
    SELECT * FROM #ejecucion_a
    UNION ALL
    SELECT * FROM #ejecucion_b
    UNION ALL
    SELECT * FROM #ejecucion_c
    UNION ALL
    SELECT * FROM #ejecucion_d
    UNION ALL
    SELECT * FROM #ejecucion_e
    UNION ALL
    SELECT * FROM #ejecucion_f
) u
"""
work_conn.execute(text(sql_ejecucion_cnf_trim_prev))

# DROP TABLE de los intermedios ya consumidos (limpieza del SAS original)
for _t in ["#ejecucion_a_prelim", "#ejecucion_a", "#ejecucion_b", "#ejecucion_c",
           "#ejecucion_c_prelim", "#ejecucion_d", "#ejecucion_e", "#ejecucion_e_prelim",
           "#ejecucion_f"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# /*Incorporacion de imputaciones permanentes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cnf_dip_aper_imput_perm"))
sql_ejec_cnf_dip_aper_imput_perm = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       (CASE WHEN t1.C_ACT = 'FFAA' THEN 'AAPP'
             WHEN t1.C_ACT = 'FFOO' THEN 'AAPP'
             WHEN t1.C_ACT = 'PREVISION' THEN 'AAPP'
             WHEN t1.C_ACT = 'FONASA' THEN 'AAPP'
             WHEN t1.C_ACT = 'CORFO' THEN 'AAPP'
             WHEN t1.C_ACT = 'CENABAST' THEN 'AAPP'
             WHEN t1.C_ACT = 'AAPP' THEN 'AAPP'
             WHEN t1.C_ACT = 'SSPP' THEN 'SSPP'
             WHEN t1.C_ACT = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       t1.Dato AS Dato
INTO #ejec_cnf_dip_aper_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER t1
WHERE t1.AÑO = {int(Year)}
  AND t1.C_SCN NOT IN ('F22','F29','F3','F32','F41','F42','F5','F612','F7','F71','F79','K10','K11','K9')
"""
work_conn.execute(text(sql_ejec_cnf_dip_aper_imput_perm))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_trim_aper"))
sql_ejecucion_cnf_trim_aper = """
SELECT * INTO #ejecucion_cnf_trim_aper
FROM (
    SELECT * FROM #ejecucion_cnf_trim_prev
    UNION ALL
    SELECT * FROM #ejec_cnf_dip_aper_imput_perm
) u
"""
work_conn.execute(text(sql_ejecucion_cnf_trim_aper))

# /*Reasignación de P3 para cuentas de subsector */
res = work_conn.execute(text("UPDATE #ejecucion_cnf_trim_aper SET C_SCN = 'D73' WHERE C_SCN = 'P3'"))
_log("UPDATE #ejecucion_cnf_trim_aper P3->D73", res.rowcount)

# /*Reasignación de nivel regional a central, por metodología de acuerdo a caracteristicas
# actuales de GOREs */
res = work_conn.execute(text("UPDATE #ejecucion_cnf_trim_aper SET NIVEL = 'CENTRAL' WHERE NIVEL = 'REGIONAL'"))
_log("UPDATE #ejecucion_cnf_trim_aper REGIONAL->CENTRAL", res.rowcount)

# /* Reasignación FBCF realizada por municipalidades pero medida en ejecución GC */
res = work_conn.execute(text(
    "UPDATE #ejecucion_cnf_trim_aper SET NIVEL = 'MUNICIPAL' "
    "WHERE NIVEL = 'CENTRAL' AND C_SCN = 'P51' AND C_CAGENTE = 'S1313'"
))
_log("UPDATE #ejecucion_cnf_trim_aper FBCF municipal", res.rowcount)


In [ ]:
# /* Eliminación de transacciones redundantes en consolidación */
for _scn, _cag in [('D73','S13'), ('D73','S131'), ('D93','S13'), ('D93','S131'), ('D73','S1312'), ('D93','S1312')]:
    res = work_conn.execute(text(
        f"DELETE FROM #ejecucion_cnf_trim_aper WHERE C_SCN = '{_scn}' AND NIVEL = 'CENTRAL' AND C_CAGENTE = '{_cag}'"
    ))
    _log(f"DELETE #ejecucion_cnf_trim_aper CENTRAL {_scn}/{_cag}", res.rowcount)

for _scn in ('D73', 'D93'):
    res = work_conn.execute(text(
        f"DELETE FROM #ejecucion_cnf_trim_aper WHERE C_SCN = '{_scn}' AND NIVEL = 'PREVISION' AND C_CAGENTE = 'S1314'"
    ))
    _log(f"DELETE #ejecucion_cnf_trim_aper PREVISION {_scn}/S1314", res.rowcount)

# DROP TABLE EJECUCION_CNF_DIP_IMPUT_PERM, EJECUCION_CNF_trim_prev del SAS original
# (el primer nombre no coincide con ninguna tabla creada en este nodo -> limpieza
# de la que solo se traduce la que sí existe)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_trim_prev"))


In [ ]:
# /*GENERA SALIDA CNF TRIMESTRAL AGREGADA - SOLO CGR*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cnf_trim"))
sql_ejecucion_cnf_trim = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, SUM(t1.DATO) AS Dato
INTO #ejecucion_cnf_trim
FROM #ejecucion_cnf_trim_aper t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
         t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_cnf_trim))
ejecucion_cnf_trim = pd.read_sql(text("SELECT * FROM #ejecucion_cnf_trim"), work_conn)
_log("ejecucion_cnf_trim", ejecucion_cnf_trim)


In [ ]:
# /* Extracción para ingresos municipales insumo para CCAS de CNF Gob General */
work_conn.execute(text("DROP TABLE IF EXISTS #ingresos_gl_a"))
sql_ingresos_gl_a = f"""
SELECT t1.SUBSECTOR, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       CAST(t1.MONEDA AS char(3)) AS MONEDA,
       t1.C_T_AREA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ANALITICO, t1.ANALITICO,
       CAST(SUM(t1.DEVENG / 1000000) AS decimal(18,2)) AS SUMADEVENG
INTO #ingresos_gl_a
FROM GOBGENER.dbo.EJECUCION_MUN t1
WHERE t1.AÑO = {int(Year)} AND t1.AÑO < 2022 AND t1.C_T_AREA IN ('GM')
GROUP BY t1.SUBSECTOR, t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END),
         t1.MONEDA, t1.C_T_AREA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ANALITICO, t1.ANALITICO
"""
work_conn.execute(text(sql_ingresos_gl_a))
ingresos_gl_a = pd.read_sql(text("SELECT * FROM #ingresos_gl_a"), work_conn)
_log("ingresos_gl_a", ingresos_gl_a)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ingresos_gl_b"))
sql_ingresos_gl_b = f"""
SELECT t1.SUBSECTOR, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MONEDA,
       (CASE WHEN t1.C_SERVICIO = 1 THEN 'GM'
             WHEN t1.C_SERVICIO = 2 THEN 'ED'
             WHEN t1.C_SERVICIO = 3 THEN 'SL'
             WHEN t1.C_SERVICIO = 4 THEN 'CM'
             ELSE '' END) AS C_T_AREA,
       (CASE WHEN t1.C_T_CUENTA = 1 THEN 'Y'
             WHEN t1.C_T_CUENTA = 2 THEN 'G'
             ELSE '' END) AS C_TIPO_CUENTA,
       t1.C_CUENTA, t1.C_ANALITICO, t1.N_ASIGNACION AS ANALITICO,
       CAST(SUM(t1.DEVENGO / 1000000) AS decimal(18,2)) AS SUMADEVENG
INTO #ingresos_gl_b
FROM GOBGENER.dbo.EJECUCION_MUN_P t1
WHERE t1.AÑO = {int(Year)} AND t1.AÑO >= 2022 AND t1.C_SERVICIO IN (1)
GROUP BY t1.SUBSECTOR, t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END),
         t1.MONEDA, t1.C_SERVICIO, t1.C_T_CUENTA, t1.C_CUENTA, t1.C_ANALITICO, t1.N_ASIGNACION
"""
work_conn.execute(text(sql_ingresos_gl_b))
ingresos_gl_b = pd.read_sql(text("SELECT * FROM #ingresos_gl_b"), work_conn)
_log("ingresos_gl_b", ingresos_gl_b)


In [ ]:
# UNE tablas GL y limpia temporales A/B (equivalente al DATA step + PROC SQL DROP)
work_conn.execute(text("DROP TABLE IF EXISTS #ingresos_gl"))
sql_ingresos_gl = """
SELECT * INTO #ingresos_gl
FROM (
    SELECT * FROM #ingresos_gl_a
    UNION ALL
    SELECT * FROM #ingresos_gl_b
) u
"""
work_conn.execute(text(sql_ingresos_gl))
ingresos_gl = pd.read_sql(text("SELECT * FROM #ingresos_gl"), work_conn)
_log("ingresos_gl", ingresos_gl)

for _t in ["#ingresos_gl_a", "#ingresos_gl_b"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))
